In [ ]:

from datetime import datetime
import numpy as np
import pandas as pd
import pickle
import yaml

import sys
sys.path.append('..')

from utils.utils import pg_engine
from utils.extract_utils import _postgres_upsert

# Let's only consider the week ahead 
"""ToDo: Consider adding safeguards to ensure predictions are for future
games here. Currently handeled only in transform. Can fail if predictions occur 
well after prediction frame is created
"""

import os
os.environ['DB_HOST'] = 'localhost'

engine = pg_engine()

query = f"""SELECT * FROM crm_mart.prediction_frame
            WHERE gamedate < CURRENT_DATE + 7
            AND gamedt > CURRENT_TIMESTAMP
        """


prediction_frame = pd.read_sql(query, con = engine )

with open('../modeling/model_config.yml', "r") as file:
        ml_config = yaml.safe_load(file)

features = ml_config['features']  

#ToDo: Think of better way to fillna
X = prediction_frame[features]
X = X.fillna(-1000)

with open('../modeling/saved_models/model.pkl','rb') as f:
    model = pickle.load(f)

predicted_prob = model.predict_proba(X.to_numpy())[:,1]
prediction_frame['home_win_prob'] = predicted_prob
prediction_frame['home_win_predicted'] = np.rint(predicted_prob)

prediction_frame['forecast_date'] = datetime.today().strftime("%Y-%m-%d")
prediction_frame['forecast_datetime'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

#Save prediction into PG
prediction_frame.to_sql(name = "completed_predictions", schema = "crm_mart", con = engine,
                        if_exists = "append", index=False, 
                        method = _postgres_upsert)

/var/folders/ml/gcby2v0941g441vx4086bdqh0000gn/T/ipykernel_15874/2378794504.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.fillna(-1000)


In [7]:
predicted_prob

array([0.66878027, 0.49306977, 0.813624  , 0.21135788, 0.7337899 ,
       0.5635075 , 0.2552713 , 0.58784443, 0.37635618, 0.23698829,
       0.28595516, 0.08733326, 0.4090434 , 0.5480563 , 0.6036091 ,
       0.6812529 , 0.87659204, 0.5761801 , 0.67967033, 0.77066046,
       0.44285625, 0.41384816, 0.5809976 , 0.5435969 , 0.2874662 ,
       0.5018908 , 0.72902584, 0.37876415, 0.58942986, 0.36323756,
       0.23687875, 0.23046492, 0.25896278, 0.25896278, 0.0996391 ,
       0.2383455 , 0.62485534, 0.264883  , 0.7100476 , 0.61800754,
       0.707609  , 0.85713893, 0.70616966, 0.46883658, 0.42385378,
       0.28256816, 0.49172863, 0.46761218, 0.2749545 , 0.2711794 ,
       0.8012857 , 0.67254794, 0.61980367, 0.83525115, 0.7232687 ,
       0.8840398 , 0.72673327, 0.34740925, 0.48019776, 0.4759485 ,
       0.25952977, 0.4363123 , 0.4759485 , 0.4221878 , 0.6250396 ,
       0.3455344 , 0.34409025, 0.20216316, 0.34635407, 0.2569529 ,
       0.4678769 , 0.8440591 , 0.6332586 , 0.3447327 , 0.64163

In [7]:
os.getcwd()

'/Users/boogleybear/Documents/pgh_hobby_projects/bucco_forecasting'